# 05 — ABSA sur 2 000 segments

GPT-OSS 20B est exécuté sur un échantillon stratifié de 2 000 segments, par blocs de 200 et avec 4 segments par requête.

In [1]:
from pathlib import Path
import sys, json, math

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
while not (ROOT / "config" / "project_config.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

assert (ROOT / "config" / "project_config.json").exists(), "Project root not found."

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

CONFIG = json.loads(
    (ROOT / "config" / "project_config.json").read_text(encoding="utf-8")
)

print("ROOT:", ROOT)

ROOT: C:\Users\wiame.bourass\Downloads\sephora-consumer-voice-intelligence-llm-first\sephora-consumer-voice-intelligence-llm-first


## 1. Échantillon stratifié

In [2]:
# Cellule 1 — Paramètres du test ciblé

from src.sampling_utils import list_segment_parts
import hashlib
import math
import pandas as pd

parts = list_segment_parts(ROOT)
assert parts, "Exécute d'abord le notebook 02."

MIN_WORDS = CONFIG["min_words_for_llm"]

N_TEST = 2000
BATCH_SIZE = 4
CHUNK_SIZE = 200
SEED = 42
MAX_SEGMENTS_PER_REVIEW = 2

TARGET_CATEGORIES = [
    "Moisturizers",
    "Treatments",
    "Cleansers",
]

# Répartition souhaitée des ratings dans l'échantillon
# Plus équilibrée que le corpus brut pour avoir assez de signaux négatifs.
RATING_TARGET = {
    "positive_4_5": 0.65,
    "neutral_3": 0.10,
    "negative_1_2": 0.25,
}


def stable_key(value, seed=SEED):
    h = hashlib.blake2b(
        f"{seed}|{value}".encode("utf-8"),
        digest_size=8,
    ).digest()

    return int.from_bytes(h, "big", signed=False) % (2**63 - 1)


def rating_group(rating):
    if pd.isna(rating):
        return "MISSING"

    rating = float(rating)

    if rating <= 2:
        return "negative_1_2"
    elif rating == 3:
        return "neutral_3"
    else:
        return "positive_4_5"

In [3]:
# Cellule 2 — Construire le pool de reviews éligibles

review_candidates = []

for p in parts:

    x = pd.read_parquet(p)

    x = x[
        (x["segment_word_count"] >= MIN_WORDS)
        & (x["secondary_category"].isin(TARGET_CATEGORIES))
    ].copy()

    if x.empty:
        continue

    # clé stable par segment
    x["_segment_key"] = (
        x["segment_id"]
        .astype(str)
        .map(stable_key)
    )

    # Maximum 2 segments éligibles par review.
    x = (
        x.sort_values(["review_id", "_segment_key"])
        .groupby("review_id", group_keys=False)
        .head(MAX_SEGMENTS_PER_REVIEW)
    )

    review_candidates.append(x)


# Fusionner les candidats de toutes les parts.
review_pool = pd.concat(
    review_candidates,
    ignore_index=True,
)

# Sécurité : une même review peut éventuellement apparaître dans plusieurs parts.

review_pool = (
    review_pool
    .sort_values(["review_id", "_segment_key"])
    .groupby("review_id", group_keys=False)
    .head(MAX_SEGMENTS_PER_REVIEW)
    .reset_index(drop=True)
)

review_pool["_rating_group"] = (
    review_pool["rating"]
    .map(rating_group)
)

review_pool["_stratum"] = (
    review_pool["secondary_category"].astype(str)
    + " | "
    + review_pool["_rating_group"]
)

print("Reviews éligibles :", f"{len(review_pool):,}")

display(
    review_pool["secondary_category"]
    .value_counts()
    .to_frame("reviews")
)

Reviews éligibles : 1,278,040


,reviews
secondary_category,
Moisturizers,526655
Treatments,409265
Cleansers,342120


In [4]:
# Cellule 3 — Quotas category × rating

# poids relatif réel des 3 catégories
category_counts = (
    review_pool["secondary_category"]
    .value_counts()
)

category_weights = (
    category_counts
    / category_counts.sum()
)

print("Poids des catégories dans le pool :")
display(
    (100 * category_weights)
    .round(2)
    .to_frame("pct")
)


quota_rows = []

for category in TARGET_CATEGORIES:

    category_target = round(
        N_TEST * category_weights.loc[category]
    )

    for rating_name, rating_share in RATING_TARGET.items():

        quota = round(
            category_target * rating_share
        )

        available = len(
            review_pool[
                (review_pool["secondary_category"] == category)
                & (review_pool["_rating_group"] == rating_name)
            ]
        )

        quota_rows.append({
            "secondary_category": category,
            "rating_group": rating_name,
            "available": available,
            "quota": min(quota, available),
        })


quota_df = pd.DataFrame(quota_rows)

quota_df["_stratum"] = (
    quota_df["secondary_category"]
    + " | "
    + quota_df["rating_group"]
)

display(quota_df)

print(
    "Quota total initial :",
    quota_df["quota"].sum()
)

Poids des catégories dans le pool :


,pct
secondary_category,
Moisturizers,41.21
Treatments,32.02
Cleansers,26.77


,secondary_category,rating_group,available,quota,_stratum
0,Moisturizers,positive_4_5,435198,536,Moisturizers | positive_4_5
1,Moisturizers,neutral_3,39883,82,Moisturizers | neutral_3
2,Moisturizers,negative_1_2,51574,206,Moisturizers | negative_1_2
3,Treatments,positive_4_5,339817,416,Treatments | positive_4_5
4,Treatments,neutral_3,28973,64,Treatments | neutral_3
5,Treatments,negative_1_2,40475,160,Treatments | negative_1_2
6,Cleansers,positive_4_5,284636,348,Cleansers | positive_4_5
7,Cleansers,neutral_3,24329,54,Cleansers | neutral_3
8,Cleansers,negative_1_2,33155,134,Cleansers | negative_1_2


Quota total initial : 2000


In [5]:
# Cellule 4 — Ajuster les quotas pour obtenir exactement 2000

difference = N_TEST - quota_df["quota"].sum()

while difference != 0:

    if difference > 0:

        candidates = quota_df[
            quota_df["quota"] < quota_df["available"]
        ]

        if candidates.empty:
            break

        for idx in candidates.index:

            quota_df.loc[idx, "quota"] += 1
            difference -= 1

            if difference == 0:
                break

    else:

        candidates = quota_df[
            quota_df["quota"] > 1
        ]

        if candidates.empty:
            break

        for idx in candidates.index:

            quota_df.loc[idx, "quota"] -= 1
            difference += 1

            if difference == 0:
                break


assert quota_df["quota"].sum() == N_TEST

display(quota_df)

print(
    "Total final :",
    quota_df["quota"].sum()
)

,secondary_category,rating_group,available,quota,_stratum
0,Moisturizers,positive_4_5,435198,536,Moisturizers | positive_4_5
1,Moisturizers,neutral_3,39883,82,Moisturizers | neutral_3
2,Moisturizers,negative_1_2,51574,206,Moisturizers | negative_1_2
3,Treatments,positive_4_5,339817,416,Treatments | positive_4_5
4,Treatments,neutral_3,28973,64,Treatments | neutral_3
5,Treatments,negative_1_2,40475,160,Treatments | negative_1_2
6,Cleansers,positive_4_5,284636,348,Cleansers | positive_4_5
7,Cleansers,neutral_3,24329,54,Cleansers | neutral_3
8,Cleansers,negative_1_2,33155,134,Cleansers | negative_1_2


Total final : 2000


In [6]:
# Cellule 5 — Sélectionner les 2000 segments

review_pool["_sample_key"] = (
    review_pool["segment_id"]
    .astype(str)
    .map(stable_key)
)

selected = []

for _, row in quota_df.iterrows():

    category = row["secondary_category"]
    rating_name = row["rating_group"]
    quota = int(row["quota"])

    g = review_pool[
        (review_pool["secondary_category"] == category)
        & (review_pool["_rating_group"] == rating_name)
    ].copy()

    g = g.nsmallest(
        quota,
        "_sample_key",
    )

    population_n = len(
        review_pool[
            (review_pool["secondary_category"] == category)
            & (review_pool["_rating_group"] == rating_name)
        ]
    )

    g["sample_weight"] = (
        population_n / len(g)
        if len(g) > 0
        else 0
    )

    selected.append(g)


test_segments = pd.concat(
    selected,
    ignore_index=True,
)

test_segments = (
    test_segments
    .sort_values("_sample_key")
    .reset_index(drop=True)
)


assert len(test_segments) == N_TEST

assert (
    test_segments.groupby("review_id").size().max()
    <= MAX_SEGMENTS_PER_REVIEW
), "Une review dépasse le nombre maximal de segments autorisé."

print(
    "Segments sélectionnés :",
    len(test_segments)
)

print(
    "Reviews distinctes :",
    test_segments["review_id"].nunique()
)

Segments sélectionnés : 2000
Reviews distinctes : 1995


## 2. Vérification de l’échantillon

In [7]:
# Cellule 6 — Contrôle de l'échantillon

print("=== CATEGORY × RATING ===")

display(
    pd.crosstab(
        test_segments["secondary_category"],
        test_segments["_rating_group"],
        margins=True,
    )
)


print("\n=== DISTRIBUTION CATÉGORIES ===")

display(
    test_segments["secondary_category"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("pct")
)


print("\n=== DISTRIBUTION RATINGS ===")

display(
    test_segments["_rating_group"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("pct")
)


print("\n=== DIVERSITÉ ===")

print(
    "Marques :",
    test_segments["brand_name_catalog"].nunique()
)

print(
    "Produits :",
    test_segments["product_id"].nunique()
)

print(
    "Reviews :",
    test_segments["review_id"].nunique()
)

print("\n=== SKIN TYPE — contrôle secondaire ===")

display(
    test_segments["skin_type"]
    .fillna("MISSING")
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("pct")
)

=== CATEGORY × RATING ===


_rating_group,negative_1_2,neutral_3,positive_4_5,All
secondary_category,,,,
Cleansers,134,54,348,536
Moisturizers,206,82,536,824
Treatments,160,64,416,640
All,500,200,1300,2000



=== DISTRIBUTION CATÉGORIES ===


,pct
secondary_category,
Moisturizers,41.2
Treatments,32.0
Cleansers,26.8



=== DISTRIBUTION RATINGS ===


,pct
_rating_group,
positive_4_5,65.0
negative_1_2,25.0
neutral_3,10.0



=== DIVERSITÉ ===
Marques : 99
Produits : 645
Reviews : 1995

=== SKIN TYPE — contrôle secondaire ===


,pct
skin_type,
combination,48.75
dry,17.40
normal,11.60
oily,11.20
MISSING,11.05


In [8]:
print("Segments :", len(test_segments))
print("Reviews distinctes :", test_segments["review_id"].nunique())

print("\nSegments par review :")
display(
    test_segments.groupby("review_id")
    .size()
    .value_counts()
    .sort_index()
    .rename_axis("segments_par_review")
    .to_frame("nombre_reviews")
)

Segments : 2000
Reviews distinctes : 1995

Segments par review :


,nombre_reviews
segments_par_review,
1,1990
2,5


## 3. Estimation du volume de tokens

In [9]:
# Cellule 7 — Paramètres d'inférence

EST_API_CALLS = math.ceil(
    len(test_segments) / BATCH_SIZE
)

N_CHUNKS = math.ceil(
    len(test_segments) / CHUNK_SIZE
)

print(f"Segments : {len(test_segments):,}")
print(f"Batch API : {BATCH_SIZE}")
print(f"Chunk size : {CHUNK_SIZE}")
print(f"Nombre de chunks : {N_CHUNKS}")
print(f"Appels API estimés : {EST_API_CALLS:,}")

Segments : 2,000
Batch API : 4
Chunk size : 200
Nombre de chunks : 10
Appels API estimés : 500


## 4. Inférence GPT-OSS 20B

Chaque bloc est sauvegardé avant de poursuivre afin de permettre la reprise après interruption.

In [10]:
# Nettoyer les colonnes techniques avant toute sauvegarde Parquet / inférence

technical_cols = ["_segment_key", "_sample_key"]
test_segments = test_segments.drop(columns=technical_cols, errors="ignore").copy()

assert "_segment_key" not in test_segments.columns
assert "_sample_key" not in test_segments.columns

print("✅ Colonnes techniques supprimées")
print("Shape :", test_segments.shape)


✅ Colonnes techniques supprimées
Shape : (2000, 22)


In [11]:
# Vérification de compatibilité Parquet avant de lancer les appels API
from pathlib import Path

_parquet_test = ROOT / "data" / "interim" / "_tmp_sample_parquet_check.parquet"
test_segments.head(20).to_parquet(_parquet_test, index=False)
_parquet_test.unlink(missing_ok=True)
print("✅ Échantillon compatible Parquet")


✅ Échantillon compatible Parquet


In [12]:
RUN_TEST = True

if RUN_TEST:
    from src.absa_utils import load_taxonomy, core_aspect_ids, build_system_prompt
    from src.llm_client import ChatCompletionsHTTPClient
    from src.inference_utils import infer_dataframe_resumable

    taxonomy = load_taxonomy(ROOT)
    aspect_ids = core_aspect_ids(taxonomy)
    prompt_version = CONFIG["llm"]["prompt_version"]
    system_prompt = build_system_prompt(taxonomy, prompt_version)

    client = ChatCompletionsHTTPClient.from_env(ROOT, CONFIG)
    print("Model:", client.model)
    assert client.model == "openai/gpt-oss-20b", (
        f"Modèle inattendu: {client.model}. Vérifie ton .env puis recrée le client/kernel."
    )

    OUT_DIR = ROOT / "data" / "interim" / "test_2000_gpt_oss_20b_batch4_chunks200_v2"
    CHUNKS_DIR = OUT_DIR / "chunks"
    RAW_BASE = OUT_DIR / "raw"
    CHECKPOINT_BASE = OUT_DIR / "checkpoints"

    for d in [OUT_DIR, CHUNKS_DIR, RAW_BASE, CHECKPOINT_BASE]:
        d.mkdir(parents=True, exist_ok=True)

    test_segments.to_parquet(OUT_DIR / "sample_2000.parquet", index=False)

    print("✅ Test activé")
    print(f"Segments totaux : {len(test_segments):,}")
    print(f"Blocs de : {CHUNK_SIZE}")
    print(f"Nombre de blocs : {N_CHUNKS}")
    print(f"Batch API interne : {BATCH_SIZE} segments/appel")
    print(f"Appels API estimés au total : {EST_API_CALLS:,}")

    all_status, all_pairs, all_requests = [], [], []

    for chunk_idx, start in enumerate(range(0, len(test_segments), CHUNK_SIZE)):
        end = min(start + CHUNK_SIZE, len(test_segments))
        chunk = test_segments.iloc[start:end].copy()
        chunk = chunk.drop(columns=["_segment_key", "_sample_key"], errors="ignore")
        chunk_tag = f"chunk_{chunk_idx:02d}"

        status_path = CHUNKS_DIR / f"{chunk_tag}_status.parquet"
        pairs_path = CHUNKS_DIR / f"{chunk_tag}_pairs.parquet"
        requests_path = CHUNKS_DIR / f"{chunk_tag}_requests.csv"

        print("\n" + "=" * 70)
        print(f"{chunk_tag}: {len(chunk)} segments")
        print(f"Appels API estimés pour ce bloc: {math.ceil(len(chunk) / BATCH_SIZE):,}")

        if status_path.exists() and pairs_path.exists() and requests_path.exists():
            print(f"SKIP {chunk_tag} — déjà terminé")
            st = pd.read_parquet(status_path)
            pr = pd.read_parquet(pairs_path)
            rq = pd.read_csv(requests_path)
        else:
            st, pr, rq = infer_dataframe_resumable(
                chunk,
                client,
                system_prompt,
                set(aspect_ids),
                batch_size=BATCH_SIZE,
                checkpoint_batches=CONFIG["llm"].get("checkpoint_batches", 25),
                temperature=CONFIG["llm"]["temperature"],
                checkpoint_dir=CHECKPOINT_BASE / chunk_tag,
                raw_dir=RAW_BASE / chunk_tag,
                run_name=f"gpt_oss_20b_test_2000_{chunk_tag}",
                prompt_version=prompt_version,
            )

            st.to_parquet(status_path, index=False)
            pr.to_parquet(pairs_path, index=False)
            rq["chunk_id"] = chunk_idx
            rq.to_csv(requests_path, index=False)

        all_status.append(st)
        all_pairs.append(pr)
        all_requests.append(rq)

        issue_count = int((st["issues_count"] > 0).sum()) if "issues_count" in st.columns else None
        print(f"✅ {chunk_tag} terminé | status={len(st):,} | pairs={len(pr):,} | issues={issue_count}")

    status = pd.concat(all_status, ignore_index=True) if all_status else pd.DataFrame()
    pairs = pd.concat(all_pairs, ignore_index=True) if all_pairs else pd.DataFrame()
    requests_log = pd.concat(all_requests, ignore_index=True) if all_requests else pd.DataFrame()

    status.to_parquet(OUT_DIR / "status.parquet", index=False)
    pairs.to_parquet(OUT_DIR / "pairs.parquet", index=False)
    requests_log.to_csv(OUT_DIR / "requests.csv", index=False)

    print("\n✅ TEST 2 000 TERMINÉ")
    print("Status rows:", len(status))
    print("Aspect pairs:", len(pairs))
    if "issues_count" in status.columns:
        print("Segments with issues:", int((status["issues_count"] > 0).sum()))
    print("Outputs:", OUT_DIR)
else:
    print("⛔ RUN_TEST=False — aucun appel API ne sera lancé.")


Model: openai/gpt-oss-20b
✅ Test activé
Segments totaux : 2,000
Blocs de : 200
Nombre de blocs : 10
Batch API interne : 4 segments/appel
Appels API estimés au total : 500

chunk_00: 200 segments
Appels API estimés pour ce bloc: 50
  wrote checkpoint block 00000: 200 segments
✅ chunk_00 terminé | status=200 | pairs=133 | issues=32

chunk_01: 200 segments
Appels API estimés pour ce bloc: 50
  wrote checkpoint block 00000: 200 segments
✅ chunk_01 terminé | status=200 | pairs=147 | issues=17

chunk_02: 200 segments
Appels API estimés pour ce bloc: 50
  wrote checkpoint block 00000: 200 segments
✅ chunk_02 terminé | status=200 | pairs=142 | issues=29

chunk_03: 200 segments
Appels API estimés pour ce bloc: 50
  wrote checkpoint block 00000: 200 segments
✅ chunk_03 terminé | status=200 | pairs=135 | issues=30

chunk_04: 200 segments
Appels API estimés pour ce bloc: 50
  wrote checkpoint block 00000: 200 segments
✅ chunk_04 terminé | status=200 | pairs=140 | issues=33

chunk_05: 200 segments


## 5. Paramètres d’exécution

- 2 000 segments au total
- blocs de 200
- 4 segments par requête